In [8]:
import requests
import pandas as pd
import time
import pyarrow
from urllib.parse import quote
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [11]:
import json
import os

with open('../data/interim/target_app_ids.json', 'w') as f:
    json.dump(lista_app_ids, f)

print("IDs guardados exitosamente. Listos para ser extraídos a la velocidad de la luz por Go!")


IDs guardados exitosamente. Listos para ser extraídos a la velocidad de la luz por Go!


In [ ]:
def recolectar_interacciones_steam(app_ids, max_resenas_por_juego=1000):
    todas_las_resenas = []
    
    for app_id in app_ids:
        cursor = '*'
        resenas_recolectadas = 0
        
        while resenas_recolectadas < max_resenas_por_juego:
            cursor_codificado = quote(cursor)
            url = f"https://store.steampowered.com/appreviews/{app_id}?json=1&filter=recent&language=english&cursor={cursor_codificado}&num_per_page=100"
            
            response = requests.get(url)
            data = response.json()
            
            if not data.get('reviews'):
                break
                
            for review in data['reviews']:
                todas_las_resenas.append({
                    'app_id': app_id,
                    'steam_id': review['author']['steamid'],
                    'playtime_forever': review['author']['playtime_forever'],
                    'voted_up': review['voted_up'],
                    'review_text': review['review']
                })
            
            cursor = data['cursor']
            resenas_recolectadas += len(data['reviews'])
            
            time.sleep(1.5)
            
    df_interacciones = pd.DataFrame(todas_las_resenas)
    df_interacciones.to_parquet('../data/processed/steam_interactions_v1.parquet', engine='pyarrow')

    return df_interacciones

In [10]:
df_games = pd.read_csv('../data/processed/steam_games_cleaned.csv')

df_clean = df_games.dropna(subset=['overall_review_%', 'overall_review_count'])
df_clean = df_clean[df_clean['overall_review_count'] > 500]

top_100_games = df_clean.sort_values(by='overall_review_%', ascending=False).head(100)
bottom_100_games = df_clean.sort_values(by='overall_review_%', ascending=True).head(100)

df_juegos_objetivo = pd.concat([top_100_games, bottom_100_games])
lista_app_ids = df_juegos_objetivo['app_id'].tolist()
print(f"🎯 Preparado para recolectar reseñas de {len(lista_app_ids)} juegos...")

df_resenas_final = recolectar_interacciones_steam(lista_app_ids, max_resenas_por_juego=1000)
print(f"✅ Extracción masiva terminada. Total reseñas extraídas: {len(df_resenas_final)}")


🎯 Preparado para recolectar reseñas de 200 juegos...


KeyboardInterrupt: 

Demoraba mucho la extracción, decidimos irnos por Go